# Tutorial 3 — Cell Scores

This tutorial computes per-cell disease scores from LDSC enrichment results and
aggregates them by cell-type groupings.

We use the **synthetic plant dataset** from `tests/test_portability.py` — the same
data that drives the portability test. It has:

- 3 000 cells, 12 factors (9 kept)
- 4 tissues, 3 lineages, 8 fine cell types, 3 cultivars
- 2 agronomic traits: `plant_height`, `grain_yield`

No real DRVI model or GWAS data is needed. Section 1 below builds a complete result
h5ad from scratch — `build_embed` → `write_result` → factor selection → synthetic
`.results` files → `attach_enrich_results` — exactly like Tutorials 1 and 2, so this
notebook can run on its own.

## 1. Build the synthetic analysis on disk

This replicates the `analysis` fixture from `tests/test_portability.py`, end to end.

In [1]:
import tempfile
from pathlib import Path

import anndata as ad
import h5py
import numpy as np
import pandas as pd

from scads_drvi.enrich.config import kept_dims, latent_stats_from_embed, select_factors
from scads_drvi.enrich.ldsc import read_results
from scads_drvi.io.artifacts import cell_metadata
from scads_drvi.io.result import (
    attach_enrich_results,
    build_embed,
    directional_loadings,
    read_result,
    write_result,
)
from scads_drvi.scores.aggregate import (
    block_order,
    eta_squared,
    group_matrix,
    summarize_by,
)
from scads_drvi.scores.cell import cs_from_z
from scads_drvi.stats import significant

TISSUES = ("leaf", "root", "stem", "flower")
LINEAGES = ("mesophyll", "epidermis", "vasculature")
FINE_TYPES = {
    "mesophyll": ("meso_1", "meso_2", "meso_3"),
    "epidermis": ("epi_1", "epi_2"),
    "vasculature": ("vasc_1", "vasc_2", "vasc_3"),
}
CULTIVARS = ("cv_alpha", "cv_beta", "cv_gamma")
TRAITS = ("plant_height", "grain_yield")
N_CELLS, N_FACTORS, N_KEPT = 3000, 12, 9

root = Path(tempfile.mkdtemp())
rng = np.random.default_rng(0)

fine = [t for lineage in LINEAGES for t in FINE_TYPES[lineage]]
lineage_of = {t: lin for lin in LINEAGES for t in FINE_TYPES[lin]}

cell_ids = np.array([f"plate{i % 5}:bc{i:05d}".encode() for i in range(N_CELLS)])
fine_codes = rng.integers(0, len(fine), size=N_CELLS).astype(np.int32)
tissue_codes = rng.integers(0, len(TISSUES), size=N_CELLS).astype(np.int32)
cultivar_codes = rng.integers(0, len(CULTIVARS), size=N_CELLS).astype(np.int32)
reads = rng.integers(500, 200_000, size=N_CELLS).astype(np.int64)

obs_path = root / "matrix.h5ad"
with h5py.File(obs_path, "w") as fh:
    obs = fh.create_group("obs")
    obs.attrs["_index"] = "cell_uid"
    obs.attrs["encoding-type"] = "dataframe"
    obs.create_dataset("cell_uid", data=cell_ids)
    for name, codes, cats in (
        ("fine_type", fine_codes, fine),
        ("tissue", tissue_codes, TISSUES),
        ("cultivar", cultivar_codes, CULTIVARS),
    ):
        group = obs.create_group(name)
        group.create_dataset("categories", data=np.array([c.encode() for c in cats]))
        group.create_dataset("codes", data=codes)
    obs.create_dataset("total_reads", data=reads)
    lineage_codes = np.array(
        [LINEAGES.index(lineage_of[fine[c]]) for c in fine_codes], dtype=np.int32
    )
    group = obs.create_group("lineage")
    group.create_dataset("categories", data=np.array([lin.encode() for lin in LINEAGES]))
    group.create_dataset("codes", data=lineage_codes)


class FakeDRVIModel:
    """Stands in for a trained ``scvi.external.DRVI`` (see Tutorial 1) — in a real
    analysis this is ``scads_drvi.factorize.model.load_fit(...)``."""

    def __init__(self, z, vanished):
        self._z = z
        self._vanished = np.asarray(vanished, dtype=bool)

    def get_latent_representation(self, adata=None, indices=None, batch_size=128):
        out = self._z
        if indices is not None:
            out = out[np.asarray(indices)]
        return out

    def set_latent_dimension_stats(
        self, embed, adata=None, datamodule=None, vanished_threshold=0.5
    ):
        embed.var["vanished"] = self._vanished
        embed.var["order"] = np.arange(embed.n_vars)
        embed.var["title"] = [f"DR {i + 1}" for i in range(embed.n_vars)]


obs_frame = cell_metadata(
    obs_path, columns=["fine_type", "tissue", "cultivar", "lineage", "total_reads"]
)
fake_adata = ad.AnnData(X=np.zeros((N_CELLS, 1), dtype=np.float32), obs=obs_frame)

z = rng.normal(0.2, 0.6, size=(N_CELLS, N_FACTORS)).astype(np.float32)
vanished = np.array([False] * N_KEPT + [True] * (N_FACTORS - N_KEPT))
model = FakeDRVIModel(z, vanished)

embed = build_embed(model, fake_adata, obs_columns=list(obs_frame.columns))
fit_path = root / "assembly_v3.h5ad"
write_result(
    fit_path, embed, provenance={"n_latent": N_FACTORS, "fit_name": "assembly_v3"}
)

# -- factor selection, exactly the way the real enrichment stage would ------------
embed = read_result(fit_path)
stats = latent_stats_from_embed(embed)
fmap = select_factors(stats, list(embed.var_names))
keep = kept_dims(fmap)
annot2dim = {f"k{i + 1}": dim for i, dim in enumerate(keep)}

arm = "assembly_v3_topfrac"
results_root = root / "enrich" / arm / "results"
for trait_index, trait in enumerate(TRAITS):
    trait_dir = results_root / trait
    trait_dir.mkdir(parents=True)
    for slot, annot in enumerate(annot2dim):
        z_score = 4.5 - 0.6 * slot + 0.3 * trait_index
        pd.DataFrame({
            "Category": [f"{annot}L2_0"],
            "Coefficient": [1e-8 * z_score],
            "Coefficient_std_error": [1e-8],
            "Coefficient_z-score": [z_score],
            "n_categories": [40],
            "total_h2": [0.11],
        }).to_csv(trait_dir / f"{annot}.results", sep="\t", index=False)

results = read_results(results_root, traits=TRAITS, annot2dim=annot2dim)
attach_enrich_results(embed, arm, results, factor_selection=fmap)
write_result(fit_path, embed, provenance=embed.uns["provenance"])

print("Synthetic result ready at:", fit_path)

Synthetic result ready at: /tmp/tmp0n_14uku/assembly_v3.h5ad


## 2. Load the finished result

Everything a real analysis produced lives in one h5ad now — `read_result()` is the
only call needed. `embed.uns["enrich"][arm]` carries the tidy per-trait results table
plus the factor-selection frame that was current when it was attached.

In [2]:
embed = read_result(fit_path)
arm_result = embed.uns["enrich"][arm]

print("n_cells:  ", embed.n_obs)
print("n_kept:   ", int(arm_result["factor_selection"]["kept"].sum()))
print("traits:   ", sorted(arm_result["results"]["trait"].unique()))
print()
print("Results columns:", arm_result["results"].columns.tolist())

n_cells:   3000
n_kept:    9
traits:    ['grain_yield', 'plant_height']

Results columns: ['Category', 'Coefficient', 'Coefficient_std_error', 'Coefficient_z-score', 'n_categories', 'total_h2', 'trait', 'annot', 'dim', 'direction', 'p_1tailed', 'fdr_q']


### Significant factors for `plant_height`

In [3]:
primary = arm_result["results"].loc[arm_result["results"]["trait"] == "plant_height"]
sig_mask = significant(primary)
print(f"{sig_mask.sum()} / {len(primary)} factors significant (FDR 5%)")
primary[sig_mask][["dim", "direction", "Coefficient_z-score", "fdr_q"]]

5 / 9 factors significant (FDR 5%)


,dim,direction,Coefficient_z-score,fdr_q
0,dim_0,combined,4.5,0.000031
1,dim_1,combined,3.9,0.000216
2,dim_2,combined,3.3,0.001450
3,dim_3,combined,2.7,0.007801
4,dim_4,combined,2.1,0.032156


## 3. Compute per-cell scores with `cs_from_z`

`cs_from_z` computes a z-weighted loading sum: for each cell it multiplies the
per-factor loadings by the z-scores (clipping negatives to zero), then sums.
`loadings` is `directional_loadings(embed, "pos")` — the ReLU'd signed latent matrix,
the *only* place a pos/neg split is ever materialized (see `scads_drvi.io.result`) —
subset to the kept dims.

The `CellScores` object carries its own null value so you never need to hardcode 0
or 1 next to an axis label.

In [4]:
loadings = directional_loadings(embed, "pos")[keep]

scores = cs_from_z(loadings, primary, model=arm, trait="plant_height")

print("kind:       ", scores.kind)  # ScoreKind.Z_WEIGHTED
print("null value: ", scores.null)  # 0.0 — don't hardcode
print("label:      ", scores.label)  # for axis labels
print("n_unscored: ", scores.n_unscored)
print()
scores.describe()

kind:        ScoreKind.Z_WEIGHTED
null value:  0.0
label:       $CS_i$ (z-weighted loading sum)
n_unscored:  0



count    3000.000000
mean        6.758899
std         3.260590
min         0.000000
25%         4.350227
50%         6.475196
75%         8.785032
max        20.636805
null        0.000000
Name: assembly_v3_topfrac:plant_height, dtype: float64

## 4. Aggregate by grouping variables

`summarize_by` computes mean ± SEM per group, dropping groups with fewer than
`min_cells` cells (and counting the drop).

In [5]:
cells_frame = embed.obs

summary = summarize_by(scores.values, cells_frame, by=["fine_type"], min_cells=20)

print(f"{len(summary)} groups, {summary.n_dropped} dropped (< 20 cells)")
print()
summary.frame.sort_values("mean", ascending=False)

8 groups, 0 dropped (< 20 cells)



,fine_type,n,n_scored,mean,median,std,sem,ci_low,ci_high
6,vasc_2,405,405,7.121127,7.083633,3.131753,0.155618,6.816122,7.426133
7,vasc_3,396,396,7.021479,6.528697,3.409119,0.171315,6.685709,7.357250
1,meso_2,356,356,6.905106,6.530209,3.329657,0.176471,6.559229,7.250984
3,epi_1,347,347,6.707643,6.323033,3.103133,0.166585,6.381143,7.034144
2,meso_3,374,374,6.698008,6.458729,3.522389,0.182138,6.341023,7.054992
4,epi_2,396,396,6.590467,6.412926,3.062026,0.153873,6.288882,6.892052
0,meso_1,358,358,6.572497,6.278220,3.186444,0.168409,6.242421,6.902572
5,vasc_1,368,368,6.409050,5.929288,3.279787,0.170971,6.073953,6.744146


### Multi-level grouping with `block_order`

When you have a hierarchy (lineage > fine type), `block_order` returns the names in
an order that keeps fine types within their lineage block — for use as the `order`
argument to a figure.

In [6]:
blocked = summarize_by(
    scores.values, cells_frame, by=["lineage", "fine_type"], min_cells=20
)
order, blocks = block_order(blocked, group="fine_type", block="lineage")

print("Ordered fine types:", order)
print()
for lineage_name, start, stop in blocks:
    print(f"  {lineage_name}: {order[start:stop]}")

Ordered fine types: ['meso_2', 'meso_3', 'meso_1', 'epi_1', 'epi_2', 'vasc_2', 'vasc_3', 'vasc_1']

  mesophyll: ['meso_2', 'meso_3', 'meso_1']
  epidermis: ['epi_1', 'epi_2']
  vasculature: ['vasc_2', 'vasc_3', 'vasc_1']


## 5. Two-dimensional group matrix

`group_matrix` computes a cell-type × tissue mean matrix — useful for the grouped
landscape figure in Tutorial 4.

In [7]:
means, counts = group_matrix(
    scores.values,
    cells_frame,
    index="fine_type",
    columns="tissue",
    min_cells=10,
    column_order=list(TISSUES),
)

print("means shape:", means.shape)
print()
means.style.background_gradient(cmap="Blues", axis=None)

means shape: (8, 4)



tissue,leaf,root,stem,flower
fine_type,,,,
meso_1,6.383878,6.746971,6.491747,6.663007
meso_2,6.836634,6.701183,7.095227,6.993520
meso_3,6.495085,6.355757,7.203340,6.725964
epi_1,7.291150,6.842521,6.355226,6.321160
epi_2,6.538617,6.972973,6.448635,6.375639
vasc_1,6.107395,6.314088,6.695526,6.579730
vasc_2,6.563262,7.062219,7.480862,7.529226
vasc_3,7.337626,7.021069,7.346739,6.429790


## 6. Measure covariate association with `eta_squared`

`eta_squared` measures how much of the score variance is explained by a categorical
grouping (one-way ANOVA effect size).

In [8]:
for col in ["fine_type", "tissue", "cultivar"]:
    rho2 = eta_squared(
        scores.values.to_numpy(),
        cells_frame[col].astype(str).to_numpy(),
    )
    print(f"  eta² ({col:12s}): {rho2:.4f}")

  eta² (fine_type   ): 0.0050
  eta² (tissue      ): 0.0007
  eta² (cultivar    ): 0.0022
